In [ ]:
import cv2
import os
import glob
import numpy as np
from collections import defaultdict

# 1. Setup your paths
# Point this to a new, messy photo of a bill you took with your phone!
query_image_path = "../data/test_images/my_messy_100_peso_photo.jpg" 
database_dir = "../data/database/sift_database/"

# 2. Initialize SIFT and the Matcher
sift = cv2.SIFT_create()
# BFMatcher with NORM_L2 is the standard for SIFT
matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False) 

def recognize_banknote(query_path, db_dir):
    print(f"--- Analyzing Query Image: {os.path.basename(query_path)} ---")
    
    # Read and process the query image
    query_img = cv2.imread(query_path)
    if query_img is None:
        print("Error: Could not load the query image.")
        return
        
    gray_query = cv2.cvtColor(query_img, cv2.COLOR_BGR2GRAY)
    kp_query, des_query = sift.detectAndCompute(gray_query, None)
    
    if des_query is None:
        print("No features found in the query image. Is it too blurry?")
        return

    # Dictionary to keep track of total matches for each banknote category
    category_votes = defaultdict(int)
    
    # Load all components from the database
    db_files = glob.glob(os.path.join(db_dir, "*.npy"))
    
    for db_file in db_files:
        # Load the descriptors for this specific component
        des_db = np.load(db_file)
        
        # Extract the base banknote name from the filename 
        # (e.g., "norm_clean_500PesosFront_comp_0.npy" -> "500PesosFront")
        filename = os.path.basename(db_file)
        # Adjust this splitting logic based on how exactly you named your files!
        category = filename.split("_comp_")[0].replace("norm_clean_", "")
        
        # 3. Perform KNN (K-Nearest Neighbors) matching
        # We ask for the top 2 matches for every point to apply the ratio test
        matches = matcher.knnMatch(des_query, des_db, k=2)
        
        # 4. Apply Lowe's Ratio Test to filter out garbage matches
        good_matches = 0
        for match_pair in matches:
            # Check if we successfully got 2 neighbors
            if len(match_pair) == 2:
                m, n = match_pair
                # If the first match is much closer than the second, it's a solid match
                if m.distance < 0.75 * n.distance:
                    good_matches += 1
                    
        # 5. Tally the votes
        # We only count it if there are at least a few solid matches (e.g., 5)
        if good_matches > 5:
            category_votes[category] += good_matches
            print(f"  Matched {good_matches} points with {category} component.")

    # 6. Determine the winner
    if not category_votes:
        print("\nRESULT: No bill recognized. Matches were below the threshold.")
    else:
        # Sort the dictionary by the highest number of matches
        sorted_results = sorted(category_votes.items(), key=lambda item: item[1], reverse=True)
        winner, top_score = sorted_results[0]
        
        print("\n" + "="*40)
        print(f"🥇 PREDICTED BANKNOTE: {winner}")
        print(f"   Total Matching Features: {top_score}")
        print("="*40)
        
        # Print runner-ups if there are any
        if len(sorted_results) > 1:
            print(f"🥈 Runner up: {sorted_results[1][0]} ({sorted_results[1][1]} matches)")

# Run the function
recognize_banknote(query_image_path, database_dir)

--- Analyzing Query Image: my_messy_100_peso_photo.jpg ---
